# VLM-DENTAL — Frontier VLMs on Kaggle Benchmarks vs the Harness

Runs **hosted (non-live) Kaggle Benchmark tasks** so the whole Kaggle model catalogue (34 vision-capable models) can be compared on the same radiographs, with the project's **own** agent loop, tools, rules/skills harness and clinical evaluator. Only the model changes, so the differences are the model's and the harness's.

**Conditions** (`dental_agent/external/conditions.py`): `kaggle_zero_shot` (no tools), `kaggle_tools` (all tools, harness off), `kaggle_tools_rules_skills` (tools + rules + skill index). The harness effect is measured as paired differences between them, per model, with bootstrap intervals. The trained checkpoint's evaluated records can be added to the same table (`EXTRA_EVALUATED`).

**How it works**
- The cohort is cut into **shards** (25 images per task). Every (shard, model) is one hosted run; a worker clones this repository, runs `run_cell` on its slice and returns **transport records** (answers + token usage). **Scoring happens here**, locally; ground truth never leaves this notebook.
- **Pause / resume.** A daily cap or an expired key pauses a run without recording an error, every answered turn is journalled, and a re-run replays them for free. The driver keeps its state in `campaign.json` (also synced to your existing private Hub repo), so a session a day or two later carries on without pushing or scheduling anything twice.
- **Persistence.** Results go to the **existing** private repos `HF_EVAL_REPO` (results) and `HF_TRACES_REPO` (hindsight data). No repository is ever created.
- **Credentials.** Put your `.env` in a **private Kaggle dataset** (a file named `env` or `.env`) and attach it to this notebook. Its values are read on import and the same dataset is attached to every hosted run, so workers get `HF_TOKEN` too (needed for the YOLO weights of the tool conditions and for pushing shards).

**Attach before running:** the private `.env` dataset, and the image dataset holding the cohort's images (section 8 can create it for you if it does not exist yet). Nothing here is spent until section 11; section 10 is a cheap gating smoke test.

**Run All** is safe: expensive steps are behind toggles in section 6, and the last cell always writes a report file, even when something failed earlier.

## 1. Platform Detection & Repository Clone

Detects Kaggle / Colab / local and shallow-clones the repository when the notebook is not already inside it. Hosted runs clone GitHub's **default branch** (not pinned), so this checkout is compared with the remote head and a difference is flagged: a hosted worker running different code would fail the identity check on import (by design).

In [ ]:
import os
import subprocess
import sys
import time
from pathlib import Path

IS_KAGGLE = os.path.exists("/kaggle") or "KAGGLE_KERNEL_RUN_TYPE" in os.environ
IS_COLAB = "google.colab" in sys.modules or "COLAB_GPU" in os.environ
PLATFORM_NAME = "Kaggle" if IS_KAGGLE else ("Colab" if IS_COLAB else "Local PC / Server")
print(f"[PLATFORM] Detected Runtime Environment: {PLATFORM_NAME}")

REPO_NAME = "VLM-DENTAL"
REPO_URL = "https://github.com/rezaxr14/VLM-DENTAL.git"

cwd = Path.cwd().resolve()
repo_root = next((p for p in [cwd, *cwd.parents] if p.name.lower() == REPO_NAME.lower() or (p / "dental_agent").is_dir()), None)
if repo_root is None and (cwd / REPO_NAME).is_dir():
    repo_root = cwd / REPO_NAME
if repo_root is None:
    print(f"[CLONE] No local repo found. Shallow cloning {REPO_URL}...")
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL], check=True)
    repo_root = cwd / REPO_NAME
    if not repo_root.is_dir():
        raise RuntimeError("git clone failed, check network / URL / permissions")

os.chdir(repo_root)
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))
print(f"[WORKSPACE] Using repository root: {Path.cwd()}")


def _git(*args):
    r = subprocess.run(["git", *args], capture_output=True, text=True, timeout=120)
    return r.stdout.strip() if r.returncode == 0 else ""


REPO_SHA = _git("rev-parse", "HEAD")
REMOTE_SHA = (_git("ls-remote", REPO_URL, "HEAD").split() or [""])[0]
DIRTY = bool(_git("status", "--porcelain", "--", "dental_agent", "scripts"))
print(f"[REPO] this checkout: {REPO_SHA[:10] or 'unknown'} | GitHub default branch: {REMOTE_SHA[:10] or 'unreachable'}")
if REPO_SHA and REMOTE_SHA and REPO_SHA != REMOTE_SHA:
    print("[REPO WARNING] hosted workers clone GitHub's default branch, which differs from this checkout. Push your code first,"
          " or results will be refused on import (identity check).")
if DIRTY:
    print("[REPO WARNING] uncommitted changes in dental_agent/ or scripts/: hosted workers will not see them.")
STATUS = {"platform": PLATFORM_NAME, "repo_sha": REPO_SHA, "remote_sha": REMOTE_SHA, "dirty": DIRTY}

## 2. Dependencies

Importing the scoring helpers pulls in the whole project package (torch, langgraph, scikit-learn, tqdm, ...), so this notebook needs the same list a hosted worker installs. `BASE_REQUIREMENTS` comes from `dental_agent/external/sharding.py` (a test keeps it in step with the real import chain). The Kaggle CLI is upgraded because the `benchmarks` commands need a recent version.

In [ ]:
import importlib
import importlib.util

INSTALL_MISSING = True


def pip(*args, quiet=True):
    print("[DEPENDENCY] pip install", *args)
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", *args], capture_output=True, text=True)
    if r.returncode != 0:
        print(r.stdout[-600:], r.stderr[-600:])
    return r.returncode == 0


if INSTALL_MISSING and (importlib.util.find_spec("yaml") is None or importlib.util.find_spec("tabulate") is None
                        or importlib.util.find_spec("dotenv") is None):
    pip("pyyaml", "python-dotenv", "tabulate")

from tabulate import tabulate
from dental_agent.external.sharding import BASE_REQUIREMENTS, TORCH_INSTALL_STEPS

IMPORT_NAME = {"pyyaml": "yaml", "pillow": "PIL", "scikit-learn": "sklearn", "python-dotenv": "dotenv"}
audit, missing = [], []
for req in BASE_REQUIREMENTS:
    mod = IMPORT_NAME.get(req, req.replace("-", "_"))
    ok = importlib.util.find_spec(mod) is not None
    audit.append([req, mod, "ok" if ok else "MISSING"])
    if not ok:
        missing.append(req)
if missing and INSTALL_MISSING:
    pip(*missing)
if importlib.util.find_spec("torchvision") is None and INSTALL_MISSING:
    for step in TORCH_INSTALL_STEPS:                                  # CPU wheel first, default wheel as the fallback
        if pip(*step["args"]):
            break
importlib.invalidate_caches()
for row in audit:
    if row[2] == "MISSING" and importlib.util.find_spec(IMPORT_NAME.get(row[0], row[0].replace("-", "_"))) is not None:
        row[2] = "installed now"
audit.append(["torch + torchvision", "torchvision", "ok" if importlib.util.find_spec("torchvision") else "MISSING"])

if INSTALL_MISSING:
    pip("-U", "kaggle")                                               # the `benchmarks` subcommands need a recent CLI
kv = subprocess.run(["kaggle", "--version"], capture_output=True, text=True)
bh = subprocess.run(["kaggle", "benchmarks", "--help"], capture_output=True, text=True)
audit.append(["kaggle CLI", (kv.stdout or kv.stderr).strip()[:40] or "not found", "ok" if bh.returncode == 0 else "NO `benchmarks` command"])
print(tabulate(audit, headers=["requirement", "import / version", "status"], tablefmt="fancy_grid"))
STATUS["dependencies_ok"] = all(r[2] in ("ok", "installed now") for r in audit)
if not STATUS["dependencies_ok"]:
    print("[DEPENDENCY WARNING] something above is missing; later cells that need it will fail with a clear error.")

## 3. Secrets Diagnostics & Hub Authentication

Importing `dental_agent` loads the repo `.env` and **any attached Kaggle dataset that holds a `.env`** (variables that are already set win, and the runner's own `MODEL_PROXY*` / `KAGGLE_*` / `LLM*` are never overridden). Kaggle Secrets fill what is still missing. Only **present / absent and the length** are shown, never a value. The Hub check reads the two existing repositories and creates nothing.

In [ ]:
import json

import dental_agent                                                   # loads .env and the attached .env dataset (config.load_env)
from dental_agent import config as dcfg

ENV_DATASETS = dcfg.discover_env_datasets()
DATASET_KEYS = {ds: sorted(dcfg.parse_env_file(path)) for ds, path in ENV_DATASETS}
print("[ENV] attached datasets holding a .env:", [ds for ds, _ in ENV_DATASETS] or "none (put your .env in a private dataset and attach it)")
REPO_ENV_KEYS = set(dcfg.parse_env_file(".env")) if Path(".env").is_file() else set()


def secret_source(key):
    for ds, keys in DATASET_KEYS.items():
        if key in keys and os.environ.get(key):
            return f".env dataset ({ds})"
    if key in REPO_ENV_KEYS and os.environ.get(key):
        return "repo .env"
    if os.environ.get(key):
        return "environment"
    if IS_KAGGLE:
        try:
            from kaggle_secrets import UserSecretsClient
            value = UserSecretsClient().get_secret(key)
            if value:
                os.environ[key] = value
                return "Kaggle Secrets"
        except Exception:
            pass
    return None


SECRETS = [("HF_TOKEN", "Hub access, shard pushes, YOLO weights for the tool conditions"), ("HF_EVAL_REPO", "existing results repo"),
           ("HF_TRACES_REPO", "existing hindsight repo"), ("HF_ARTIFACT_REPO", "models repo (YOLO weights)"),
           ("DENTEX_IMAGES_REPO", "optional image repo"), ("GITHUB_TOKEN", "optional")]
rows = []
for key, why in SECRETS:
    src = secret_source(key)
    rows.append([key, "[SET]" if src else "[NOT SET]", src or "-", len(os.environ.get(key, "")) or "-", why])
print(tabulate(rows, headers=["variable", "status", "source", "length", "used for"], tablefmt="fancy_grid"))

HF_OK = False
if os.environ.get("HF_TOKEN"):
    try:
        from huggingface_hub import HfApi, login
        login(token=os.environ["HF_TOKEN"])
        print(f"[AUTH] Hugging Face: authenticated as {HfApi().whoami()['name']}")
        HF_OK = True
    except Exception as exc:
        print(f"[AUTH WARNING] Hugging Face login failed: {type(exc).__name__}: {str(exc)[:160]}")
else:
    print("[AUTH WARNING] no HF_TOKEN: nothing is persisted to the Hub and the tool conditions cannot load the YOLO weights on workers.")

from dental_agent.external.persist import ExternalHub


def open_hub(repo_env):
    """An ExternalHub on an EXISTING repo named by the env variable, or None (never creates anything)."""
    repo = os.environ.get(repo_env)
    if not (HF_OK and repo):
        return None
    try:
        return ExternalHub.from_env(repo)
    except Exception as exc:
        print(f"[HUB WARNING] {repo_env}={repo}: {type(exc).__name__}: {str(exc)[:200]}")
        return None


from dental_agent.utils.inputs import traces_repo
os.environ.setdefault("HF_EVAL_REPO", "Reza-Nadimi/vlm-dental-evaluations")      # the project's existing results repo (as scripts/run_external_benchmark.py)
os.environ.setdefault("HF_TRACES_REPO", traces_repo())                            # the existing traces repo
RESULTS_HUB, TRACES_HUB = open_hub("HF_EVAL_REPO"), open_hub("HF_TRACES_REPO")
print(f"[HUB] results repo (HF_EVAL_REPO): {'reachable' if RESULTS_HUB else 'off'} | hindsight repo (HF_TRACES_REPO): {'reachable' if TRACES_HUB else 'off'}")
STATUS.update(env_datasets=[ds for ds, _ in ENV_DATASETS], hf_ok=HF_OK, results_hub=bool(RESULTS_HUB), traces_hub=bool(TRACES_HUB))

## 4. Kaggle CLI Credentials & Quota

The hosted-run commands need the Kaggle CLI authenticated for `benchmarks`. `kaggle benchmarks init` is run once when the quota query fails. The quota shows the daily and monthly allowance and when each refills; the campaign pauses on its own when a cap is hit, you do not have to watch it.

In [ ]:
from dental_agent.external import pause
from dental_agent.external.kaggle_cli import CLIError, KaggleCLI

CLI = KaggleCLI()


def read_quota():
    try:
        return pause.parse_quota(CLI.quota())
    except Exception as exc:
        return exc


QUOTA = read_quota()
if isinstance(QUOTA, Exception):
    print(f"[KAGGLE] quota query failed ({str(QUOTA)[:120]}); running `kaggle benchmarks init -y` once")
    subprocess.run(["kaggle", "benchmarks", "init", "-y", "--env-file", "/tmp/kb_init.env"], capture_output=True, text=True, timeout=300)
    QUOTA = read_quota()
if isinstance(QUOTA, Exception):
    print(f"[KAGGLE WARNING] the Kaggle CLI is not usable for benchmarks: {str(QUOTA)[:300]}")
    STATUS["kaggle_cli_ok"] = False
else:
    print(tabulate([[p, q["remaining"], q["used"], q["total"], q["refill_at"]] for p, q in QUOTA.items()],
                   headers=["period", "remaining $", "used $", "total $", "refills at"], tablefmt="fancy_grid"))
    STATUS["kaggle_cli_ok"] = True
    STATUS["quota"] = QUOTA

## 5. Hosted Model Catalogue

The catalogue was measured on real hosted runs (`dental_agent/external/model_profiles.py`): `vision = ok` models answered a probe image correctly; `unreliable`, `text_only` and `unserved` ones are left out unless you ask. Prices are indicative (per million tokens, from one tiny request) and only used for the rough estimate below.

In [ ]:
from dental_agent.external.model_profiles import HOSTED_CATALOG, HOSTED_VERIFIED_ON, hosted_model, hosted_vision_models

print(f"[CATALOGUE] verified {HOSTED_VERIFIED_ON}: {len(HOSTED_CATALOG)} listed, {len(hosted_vision_models())} vision-capable")
print(tabulate([[m.cli_slug, m.vision, m.usd_per_mtok_in, m.usd_per_mtok_out] for m in HOSTED_CATALOG],
               headers=["model (CLI slug)", "vision", "$/M in", "$/M out"], tablefmt="simple"))

## 6. Settings (the control centre)

Every value that changes a result or spends money is here. Nothing expensive runs by accident: the campaign toggles default to the safe side except the smoke test.

- **SPLIT**: `official_test250` = the 250-image test cohort (cohort prompts naming its eight classes, `test250_scoring_v2`, COCO/mapping versions in every result's identity; read from local disk, an attached Kaggle dataset, the Hugging Face cache or `DENTEX_TEST_REPO`, in that order; attach the dataset made by `scripts/stage_official_test250_dataset.py` so hosted workers can see the images); `test` = the official 50-image test split; `dev` = the dev images held out of SFT/GRPO.
- **MODELS**: `"all-vision"` or an explicit list of CLI slugs. **EXTRA_EVALUATED** adds the trained checkpoint's (or any) evaluated records to the table.
- **Pause / resume**: just re-run the campaign cell later; `WAIT_MINUTES` only bounds how long one cell blocks.

In [ ]:
# ----------------------------------------------------------------------------- WHAT TO RUN
SPLIT = "official_test250"                        # "official_test250" | "test" | "dev"
DEV_SPLIT_FILE = "data/splits/dentex_dev.json"   # only used when SPLIT == "dev"
DATA_DIR = "data"
LIMIT_IMAGES = None                              # first N images only (debugging); None = the whole split
CONDITIONS = ["kaggle_zero_shot", "kaggle_tools", "kaggle_tools_rules_skills"]
MODELS = "all-vision"                            # "all-vision" or a list of CLI slugs, e.g. ["claude-haiku-5-5", "gemini-3.8-flash"]
EXCLUDE_MODELS = []
INCLUDE_UNRELIABLE = False
PLUGINS = []                                     # modules that register extra conditions / tool packs / evaluators (segmentation, image-editing tools, ...)

# ----------------------------------------------------------------------------- HOW TO RUN
SHARD_SIZE = 25                                  # images per hosted task
WORKERS = 4                                      # images processed in parallel inside one run (the proxy accepts 16)
MAX_INFLIGHT = 60                                # hosted runs scheduled at the same time
WAIT_MINUTES = 90                                # how long ONE run of the campaign cell blocks; re-run to continue
POLL_SECONDS = 60
ACCOUNT_LABEL = os.environ.get("KAGGLE_USERNAME") or "default"   # separates the campaign state of different Kaggle accounts on the Hub

# ----------------------------------------------------------------------------- TOGGLES
CREATE_IMAGE_DATASET = False                     # section 8: stage the cohort's images as a private Kaggle dataset (once)
IMAGE_DATASET_ID = "rezanadimikj/dentex-test-images"   # owner/slug of the image dataset to create (section 8 only)
RUN_SMOKE_TEST = True                            # section 10: 1 cheap model x 2 images x all conditions; the campaign needs it to pass
SMOKE_MODEL = None                               # None = the cheapest vision model of the catalogue
SMOKE_WAIT_MINUTES = 40
RUN_CAMPAIGN = False                             # section 11: push, schedule, wait, collect (THIS SPENDS QUOTA)
RESTORE_FROM_HUB = True                          # pull what workers / earlier sessions pushed (and the campaign state) before starting
PUSH_TO_HUB = True                               # push collected results and the campaign state
RUN_HINDSIGHT_EXPORT = False                     # section 15: export every episode (successes and failures) to HF_TRACES_REPO
N_BOOT = 2000
EXTRA_EVALUATED = {}                             # {"ours/final_grpo_harness": "path/to/evaluated.jsonl"} -> added to the comparison table

ROOT = Path("/kaggle/working/kaggle_hosted") if IS_KAGGLE else Path("results/kaggle/hosted")
OUT = ROOT / "results"
for _p in (ROOT, OUT):
    _p.mkdir(parents=True, exist_ok=True)

from dental_agent.external.plugins import load_plugins
from dental_agent.external.conditions import CONDITIONS as ALL_CONDITIONS, get_condition
load_plugins(PLUGINS)
unknown = [c for c in CONDITIONS if c not in ALL_CONDITIONS]
assert not unknown, f"unknown conditions {unknown}; known: {sorted(ALL_CONDITIONS)}"
if MODELS == "all-vision":
    MODEL_LIST = [m for m in hosted_vision_models(INCLUDE_UNRELIABLE) if m not in EXCLUDE_MODELS]
else:
    MODEL_LIST = [m for m in MODELS if m not in EXCLUDE_MODELS]
    for m in MODEL_LIST:
        hosted_model(m)                           # raises UnknownModel with the verification date when it is not in the catalogue
print(f"[SETTINGS] split={SPLIT} | conditions={CONDITIONS} | {len(MODEL_LIST)} models | shard={SHARD_SIZE} | root={ROOT}")
STATUS.update(split=SPLIT, conditions=CONDITIONS, n_models=len(MODEL_LIST))

## 7. Dataset Resolution & Preflight

The cohort comes from the project's own loaders (`harness.notebook_eval.prepare_inputs` for the 250-image cohort, then `scripts/evaluate_models.load_eval_split`), so it is exactly what the other evaluations use (the cohort's settings come back with it and become part of every result's identity), and the images the loader found are matched to the **attached Kaggle datasets** they live in: the `owner/name` ids and relative paths are read back from where they are mounted, nothing is typed. Hosted workers can only see attached datasets, so an image that is not inside one is reported here, before anything is pushed. Preflight checks that the prompts render and the images open.

In [ ]:
from dental_agent.external import preflight as PF
from dental_agent.external.cohort import load_split
from dental_agent.external.sharding import locate_image_datasets
from dental_agent.utils.inputs import kaggle_dataset_dirs

ELIGIBLE, IDS, annots_df, cats_df, COHORT = None, [], None, None, None
IMAGE_DATASETS, REL_PATHS, IMAGES_ON_KAGGLE = [], {}, False
try:
    from scripts import evaluate_models as ev
    images_df, annots_df, cats_df, COHORT = load_split(SPLIT, dataset="dentex", data_dir=DATA_DIR, dev_split=DEV_SPLIT_FILE)
    ELIGIBLE = ev.eligible_images(images_df, LIMIT_IMAGES)
    IDS = sorted(int(i) for i in ELIGIBLE["id"])
    gt = annots_df[annots_df["image_id"].isin(IDS)]
    print(f"[COHORT] split={SPLIT}: {len(IDS)} images with a readable file ({len(images_df)} in the split); "
          f"{gt['image_id'].nunique()} have findings, {len(gt)} findings in total")
    if COHORT:
        print(f"[COHORT] official_test250 settings: COCO {COHORT['coco_version']}, mapping {COHORT['mapping_version']} ({COHORT['mapping_sha256'][:12]}), scoring {COHORT['scoring']}")
    STATUS.pop("cohort_error", None)
except Exception as exc:
    STATUS["cohort_error"] = f"{type(exc).__name__}: {str(exc)[:300]}"
    print(f"[COHORT ERROR] the {SPLIT} split could not be loaded: {STATUS['cohort_error']}\n"
          "               Attach/download the DENTEX data as the other notebooks do (see scripts/download_and_cleanup.py), then re-run from here.")

if IDS:
    PREFLIGHT = {"prompts": PF.check_prompts_render(SPLIT), "images": PF.check_images(ELIGIBLE, IDS)}
    print("[PREFLIGHT]", {k: ("ok" if v.get("ok") else v) for k, v in PREFLIGHT.items()})
    STATUS["preflight_ok"] = all(v.get("ok") for v in PREFLIGHT.values())
    try:
        IMAGE_DATASETS, REL_PATHS = locate_image_datasets(ELIGIBLE, IDS, kaggle_dataset_dirs())
        IMAGES_ON_KAGGLE = True
        print(f"[IMAGES] hosted workers will mount: {IMAGE_DATASETS}")
    except ValueError as exc:
        print(f"[IMAGES] not usable by hosted runs yet: {exc}\n         " + ("Attach the cohort's Kaggle dataset (scripts/stage_official_test250_dataset.py + `kaggle datasets create`), then re-run from section 1."
              if SPLIT == "official_test250" else "Section 8 can stage them as a private Kaggle dataset."))
else:
    STATUS["preflight_ok"] = False
STATUS.update(n_images=len(IDS), images_on_kaggle=IMAGES_ON_KAGGLE, image_datasets=IMAGE_DATASETS)

## 8. Create the Image Dataset (once, optional)

When the cohort's images are not in an attached Kaggle dataset (the 50-image official test images; **not** the 250-image cohort, which has its own staging script, `scripts/stage_official_test250_dataset.py`, that also carries its COCO and mapping), this stages **only the images** (never annotations) as a **private** dataset and publishes it through the Kaggle CLI. Then attach it to this notebook (*Add Input → Your datasets*) and re-run from section 1; the next run finds it in section 7. Set `CREATE_IMAGE_DATASET = True` and `IMAGE_DATASET_ID` in section 6 first.

In [ ]:
if CREATE_IMAGE_DATASET and ELIGIBLE is None:
    print("[DATASET] cannot stage: the cohort did not load (section 7)")
elif CREATE_IMAGE_DATASET:
    from dental_agent.external.image_dataset import stage_image_dataset
    staged = stage_image_dataset(ELIGIBLE, IDS, ROOT / "image_dataset", dataset_id=IMAGE_DATASET_ID, title=f"DENTEX {SPLIT} images")
    how = CLI.dataset_publish(staged["dir"], IMAGE_DATASET_ID, f"{staged['images']} {SPLIT} images")
    print(f"[DATASET] {IMAGE_DATASET_ID} {how} ({staged['images']} images, private).")
    print("          Attach it to this notebook and re-run from section 1.")
    STATUS["image_dataset_created"] = IMAGE_DATASET_ID
else:
    print("[DATASET] skipped (CREATE_IMAGE_DATASET = False)" + ("" if IMAGES_ON_KAGGLE else " — but the images are not on Kaggle yet, see section 7"))

## 9. Campaign Plan & Resume

Builds the campaign (`conditions × shards` tasks, one run per model, **nothing is pushed or spent here**), prints a rough cost estimate (assumed turns, indicative prices; the real cost is in section 16) and, in a new session, **adopts the campaign state and the finished results from the Hub**, so nothing is pushed or scheduled twice.

In [ ]:
from dental_agent.external import cost_guard
from dental_agent.external.hosted_driver import HostedCampaign, build_campaign

CAMPAIGN = None
try:
    if not IDS:
        raise RuntimeError("no cohort (section 7)")
    CAMPAIGN = build_campaign(ROOT, ELIGIBLE, conditions=CONDITIONS, models=MODEL_LIST, split=SPLIT, image_ids=IDS, shard_size=SHARD_SIZE,
                              workers=WORKERS, hf_results_repo=os.environ.get("HF_EVAL_REPO"), plugins=PLUGINS, cohort_settings=COHORT)
except Exception as exc:
    print(f"[PLAN] no campaign can be built yet: {type(exc).__name__}: {str(exc)[:300]}\n       The sections that need it are skipped; fix the cause above and re-run.")
STATUS["campaign_built"] = CAMPAIGN is not None
if CAMPAIGN is not None:
    n_tasks = len([n for n in CAMPAIGN.specs if n not in CAMPAIGN.state["remainders"]])
    print(f"[PLAN] {len(IDS)} images -> {n_tasks} tasks ({len(CONDITIONS)} conditions x {-(-len(IDS) // SHARD_SIZE)} shards) x {len(MODEL_LIST)} models "
          f"= {n_tasks * len(MODEL_LIST)} hosted runs; {MAX_INFLIGHT} at a time")
    print(f"[PLAN] workers attach: {sorted({tuple(s.attach) for s in CAMPAIGN.specs.values()})}")
    est = {m: sum((cost_guard.estimate(c, len(IDS), price=hosted_model(m).price)["expected"]["cost_usd"] or 0.0) for c in CONDITIONS) for m in MODEL_LIST}
    top = sorted(est.items(), key=lambda kv: -kv[1])
    print(f"[ESTIMATE] indicative total for all models: ${sum(est.values()):,.0f} (assumes ~12 turns per tools image); most expensive: "
          + ", ".join(f"{m} ${v:,.0f}" for m, v in top[:4]))
    if RESTORE_FROM_HUB and RESULTS_HUB:
        if CAMPAIGN.restore_state_from_hub(RESULTS_HUB, ROOT / "_hub", ACCOUNT_LABEL):
            print("[HUB] adopted the campaign state of an earlier session; nothing will be pushed or scheduled twice")
        print("[HUB] results restored from the Hub:", CAMPAIGN.restore_from_hub(RESULTS_HUB, OUT, ROOT / "_hub") or "nothing new")

## 10. Gating Smoke Test (cheap)

Before the campaign, **one cheap model** runs **two images through every condition** as real hosted tasks. This verifies, on Kaggle itself and for about a cent, everything a local test cannot: the worker installs the project, imports it, mounts the dataset and the `.env`, loads the YOLO weights (tool conditions), talks to the proxy, returns outputs, and the results score locally with the identity the notebook expects. The campaign cell refuses to start while this has failed. Re-running the cell is free (state is kept).

In [ ]:
def cheapest_vision(models):
    priced = [hosted_model(m) for m in models if hosted_model(m).price]
    return min(priced, key=lambda m: m.usd_per_mtok_in + m.usd_per_mtok_out).cli_slug


def run_table(camp):
    rows = []
    for key, run in sorted(camp.state["runs"].items()):
        m = run.get("manifest") or {}
        task, model = key.split("|")
        if model not in camp.task_models.get(task, []):
            continue
        rows.append([camp.specs[task].condition.removeprefix("kaggle_"), model, run.get("outcome") or run.get("status"),
                     m.get("completed_images"), m.get("stage") or "", (m.get("error") or run.get("error") or "")[:110]])
    return rows


SMOKE_OK, SMOKE = None, None
CAMPAIGN_ALREADY_STARTED = CAMPAIGN is not None and bool(CAMPAIGN.state["tasks"])
if CAMPAIGN_ALREADY_STARTED:
    SMOKE_OK = True
    print("[SMOKE] not needed: this campaign was started in an earlier session, and the campaign only starts after a passing smoke test.")
elif not RUN_SMOKE_TEST:
    print("[SMOKE] skipped (RUN_SMOKE_TEST = False). The campaign cell will not start unless SMOKE_OK is overridden.")
elif not (IMAGES_ON_KAGGLE and STATUS.get("kaggle_cli_ok")):
    SMOKE_OK = False
    print("[SMOKE] cannot run: " + ("the images are not in an attached Kaggle dataset (sections 7-8)" if not IMAGES_ON_KAGGLE else "the Kaggle CLI is not usable (section 4)"))
else:
    SMOKE_MODEL_USED = SMOKE_MODEL or cheapest_vision(hosted_vision_models())
    SMOKE = build_campaign(ROOT / "smoke", ELIGIBLE, conditions=CONDITIONS, models=[SMOKE_MODEL_USED], split=SPLIT, image_ids=IDS[:2],
                           shard_size=2, workers=2, plugins=PLUGINS, cohort_settings=COHORT)
    print(f"[SMOKE] model={SMOKE_MODEL_USED} | tasks={len(SMOKE.specs)} | env dataset={sorted({s.env_dataset for s in SMOKE.specs.values()})}")
    print("[SMOKE] push:", SMOKE.push_missing())
    print("[SMOKE] schedule:", SMOKE.schedule(MAX_INFLIGHT))
    print("[SMOKE] waiting (re-run this cell to keep waiting):", SMOKE.wait(poll_s=30, timeout_s=SMOKE_WAIT_MINUTES * 60))
    print("[SMOKE] collect:", SMOKE.collect(ROOT / "smoke_results"))
    print(tabulate(run_table(SMOKE), headers=["condition", "model", "outcome", "images", "stage", "error"], tablefmt="fancy_grid"))
    SMOKE_RES = SMOKE.evaluate(ROOT / "smoke_results", annots_df, cats_df)
    SMOKE_OK = len(SMOKE_RES) == len(CONDITIONS) and all(len(r["records"]) == 2 and not r["errors"] for r in SMOKE_RES.values())
    print("[SMOKE]", "PASSED: hosted runs work end to end for every condition." if SMOKE_OK else
          "FAILED or not finished: read the table above (stage/error); the campaign stays blocked.")
STATUS["smoke_ok"] = SMOKE_OK

## 11. The Campaign (push, schedule, wait, collect, repeat)

`build_campaign` makes `conditions × shards` tasks and one run per model. This cell is **idempotent and resumable**:

- tasks already pushed are not pushed again, runs already scheduled are not scheduled again (state in `campaign.json`, synced to the Hub);
- finished runs are downloaded and merged; a model whose shards were **paused** by a cap or an expired key keeps what it finished, and the images it still lacks become small *remainder* tasks for that model only;
- when a **daily quota** is hit the cell stops and says when it refills; run it again then (a day or two later is fine, in a new session too).

Set `RUN_CAMPAIGN = True` in section 6 to start.

In [ ]:
def drive(minutes):
    """Push what is missing, schedule up to MAX_INFLIGHT runs, collect what finished, plan remainders; repeat until done or out of time."""
    deadline = time.time() + minutes * 60
    while True:
        pushed = CAMPAIGN.push_missing()
        sched = CAMPAIGN.schedule(MAX_INFLIGHT)
        counts = CAMPAIGN.refresh()
        tally = CAMPAIGN.collect(OUT)
        new = CAMPAIGN.plan_remainders(OUT)
        paused = CAMPAIGN.paused_until()
        print(f"[{time.strftime('%H:%M:%S')}] runs={counts} | scheduled now={sched['scheduled']} rejected={sched['rejected']} "
              f"waiting for a slot={sched['skipped_cap']} | collected={tally} | new remainder tasks={len(new)}")
        if PUSH_TO_HUB and RESULTS_HUB:
            try:
                CAMPAIGN.sync_state_to_hub(RESULTS_HUB, ROOT / "_hub", ACCOUNT_LABEL)
            except Exception as exc:
                print(f"[HUB WARNING] campaign state not synced: {type(exc).__name__}: {str(exc)[:160]}")
        quota_stop = [p for p in paused if p["kind"] in ("quota", "rate_limit")]
        if quota_stop:
            print("[PAUSED] a hosted run stopped on a cap:", {p["run"]: (p["kind"], p.get("refill_at")) for p in quota_stop[:3]},
                  "\n         Run this cell again after the allowance refills; finished images are kept and answered turns replay for free.")
            return "paused"
        if not CAMPAIGN.inflight() and not new and not sched["scheduled"] and not sched["skipped_cap"]:
            return "done"
        if time.time() > deadline:
            print(f"[TIME] {minutes} minutes used; run this cell again to continue.")
            return "time"
        time.sleep(POLL_SECONDS)


CAMPAIGN_STATE = "not started"
if not RUN_CAMPAIGN:
    print("[CAMPAIGN] not started (RUN_CAMPAIGN = False in section 6).")
elif CAMPAIGN is None:
    print("[CAMPAIGN] cannot start: no campaign was built (section 9).")
    CAMPAIGN_STATE = "no campaign"
elif SMOKE_OK is not True:
    print(f"[CAMPAIGN] blocked: the gating smoke test has not passed (SMOKE_OK={SMOKE_OK}). Fix section 10 first.")
    CAMPAIGN_STATE = "blocked by smoke test"
else:
    CAMPAIGN_STATE = drive(WAIT_MINUTES)
STATUS["campaign"] = CAMPAIGN_STATE

## 12. Status & Failures

How far every (model, condition) cell is, which runs are still active, which paused and when they can continue, and — for every run that did not finish cleanly — the **stage and error the worker reported** (install, import, preflight, grounder, env, runtime). Nobody has to read logs.

In [ ]:

if CAMPAIGN is None:
    print('[SKIPPED] no campaign was built (section 9)')
    REPORT = None
else:
    REPORT = CAMPAIGN.report(OUT)
    print("[RUNS]", REPORT["runs"], "| tasks:", REPORT["tasks"])
    conds = sorted({c.split("/")[1] for c in REPORT["cells"]})
    matrix = [[m] + [REPORT["cells"].get(f"{m}/{c}", "-") for c in conds] for m in MODEL_LIST]
    print(tabulate(matrix, headers=["model"] + [c.removeprefix("kaggle_") for c in conds], tablefmt="simple"))
    if REPORT["paused"]:
        print("[PAUSED]", REPORT["paused"][:5])
    bad = [r for r in run_table(CAMPAIGN) if r[2] not in ("complete", "completed", "scheduled", "running", "queued", "pending")]
    if bad:
        print(f"[FAILURES] {len(bad)} runs did not finish cleanly:")
        print(tabulate(bad[:40], headers=["condition", "model", "outcome", "images", "stage", "error"], tablefmt="fancy_grid"))
    STATUS["cells_done"] = REPORT["cells"]

## 13. Hub Sync

Pushes every collected cell's finished records and the campaign state to the **existing** private results repo (workers with the `.env` dataset already push their own shards; this covers the rest and lets another session or machine resume). Pushes are atomic, identity-checked and never overwrite a longer or conflicting copy.

In [ ]:

if CAMPAIGN is None:
    print('[SKIPPED] no campaign was built (section 9)')
    pass
else:
    if PUSH_TO_HUB and RESULTS_HUB:
        try:
            pushed = CAMPAIGN.push_to_hub(RESULTS_HUB, OUT, ROOT / "_hub")
            CAMPAIGN.sync_state_to_hub(RESULTS_HUB, ROOT / "_hub", ACCOUNT_LABEL)
            print(f"[HUB] pushed {len(pushed)} cells:", pushed)
            STATUS["hub_pushed_cells"] = len(pushed)
        except Exception as exc:
            print(f"[HUB WARNING] push failed (results are still on disk and in the hosted outputs): {type(exc).__name__}: {str(exc)[:300]}")
    else:
        print("[HUB] push skipped (" + ("PUSH_TO_HUB = False" if not PUSH_TO_HUB else "no results repo / token") + ")")

## 14. Scoring & the Comparison Table

Every collected cell is scored **locally** with the project's clinical evaluator (set-level matching of every finding, the exact `evaluate_condition` schema) after its identity is checked against what this checkout expects: a worker that ran different prompts, rules or tools is refused, not mixed in. The table uses `matrix.summarize_cell` and the paired bootstrap unchanged, so a model's tools and harness effect (`tools vs zero-shot`, `rules+skills vs tools`) comes with a confidence interval. `EXTRA_EVALUATED` adds the trained checkpoint's records.

In [ ]:
from dental_agent.evaluation.matrix import load_records
from dental_agent.external import tables
if CAMPAIGN is None:
    print('[SKIPPED] no campaign was built (section 9)')
    EVALUATED, CELLS, ROWS, EFFECTS, COMPARISON_MD = {}, {}, [], [], '(no campaign was built)'
else:

    EVALUATED = CAMPAIGN.evaluate(OUT, annots_df, cats_df)
    CELLS = {label: res["records"] for label, res in EVALUATED.items() if res["records"]}
    for label, path in EXTRA_EVALUATED.items():
        CELLS[label] = load_records(path)
    incomplete = {k: f"{len(v['records'])}/{len(IDS)}" for k, v in EVALUATED.items() if len(v["records"]) < len(IDS)}
    print(f"[SCORING] {len(CELLS)} cells scored; {len(incomplete)} not complete yet" + (f": {dict(list(incomplete.items())[:6])} ..." if incomplete else ""))

    ROWS = tables.table_rows(CELLS, n_boot=N_BOOT) if CELLS else []
    PAIRS = [p for m in MODEL_LIST for p in tables.standard_pairs(m)]
    EFFECTS = tables.harness_effect(CELLS, PAIRS, n_boot=N_BOOT) if CELLS else []
    COMPARISON_MD = (tables.markdown_table(sorted(ROWS, key=lambda r: -(r["exact_f1"] or 0))) + "\n" + tables.markdown_effects(EFFECTS)) if ROWS else "(no scored cells yet)"
    (ROOT / "comparison.md").write_text(COMPARISON_MD, encoding="utf-8")
    if ROWS:
        top = sorted(ROWS, key=lambda r: -(r["exact_f1"] or 0))[:15]
        print(tabulate([[r["cell"], r["n"], f"{r['format_ok']:.0%}", r["fdi_f1"], r["exact_f1"], r["exact_precision"], r["exact_recall"], r["mean_turns"]] for r in top],
                       headers=["cell", "n", "format ok", "FDI F1", "exact F1", "exact P", "exact R", "turns"], floatfmt=".3f", tablefmt="simple"))
    if EFFECTS:
        print("\nHarness effect (exact F1, paired):")
        print(tabulate([[e["label"], e["n_paired"], f"{e['diff']:+.3f}", f"[{e['ci_low']:+.3f}, {e['ci_high']:+.3f}]", f"{e['p_value']:.3f}"] for e in EFFECTS[:40]],
                       headers=["comparison", "n", "diff", "95% CI", "p"], tablefmt="simple"))
    STATUS["scored_cells"] = len(CELLS)

## 15. Hindsight Export (explicit)

Every episode (successes **and** failures: trajectory turn by turn, tools, harness events, the answer, the ground truth, an `outcome` label and the token usage) can be exported to the existing private traces repo for hindsight-RAG training. It is **off by default** (`RUN_HINDSIGHT_EXPORT`), goes to `HF_TRACES_REPO`, and never creates a repository.

In [ ]:
if RUN_HINDSIGHT_EXPORT:
    from dental_agent.external.hindsight import export_hindsight
    if TRACES_HUB is None:
        print("[HINDSIGHT] no reachable HF_TRACES_REPO / HF_TOKEN: nothing exported (see section 3)")
    else:
        summary = {}
        for label, res in EVALUATED.items():
            if res["records"]:
                model, condition = label.split("/")
                summary[label] = export_hindsight(TRACES_HUB, ROOT / "_hs", CAMPAIGN.expected_settings(model, condition), res["records"],
                                                  complete=len(res["records"]) == len(IDS))
        print(f"[HINDSIGHT] exported {sum(v['rows'] for v in summary.values())} episodes from {len(summary)} cells")
        STATUS["hindsight_rows"] = sum(v["rows"] for v in summary.values())
else:
    print("[HINDSIGHT] skipped (RUN_HINDSIGHT_EXPORT = False)")

## 16. Usage & Cost

What the hosted runs actually consumed, from the per-request usage every episode carries. A total shows as unknown (`n/a`) when any request lacked the number, never as zero.

In [ ]:
def usage_totals(records):
    tot = {"requests": 0, "prompt": 0, "completion": 0, "cost": 0.0}
    unknown = {"prompt": False, "completion": False, "cost": False}
    for r in records:
        u = (r.get("external") or {}).get("usage") or {}
        tot["requests"] += u.get("requests") or 0
        for key, field in (("prompt", "prompt_tokens"), ("completion", "completion_tokens"), ("cost", "cost_usd")):
            if u.get(field) is None:
                unknown[key] = True
            else:
                tot[key] += u[field]
    return tot, unknown


USAGE = {}
for label, recs in CELLS.items():
    if label in EVALUATED:
        USAGE[label] = usage_totals(recs)
if USAGE:
    fmt = lambda v, bad, d=0: "n/a" if bad else f"{v:,.{d}f}"
    print(tabulate([[k, len(CELLS[k]), t["requests"], fmt(t["prompt"], u["prompt"]), fmt(t["completion"], u["completion"]), fmt(t["cost"], u["cost"], 2)]
                    for k, (t, u) in sorted(USAGE.items(), key=lambda kv: -kv[1][0]["cost"])[:30]],
                   headers=["cell", "images", "requests", "prompt tok", "completion tok", "cost $"], tablefmt="simple"))
    known = [t["cost"] for t, u in USAGE.values() if not u["cost"]]
    print(f"[COST] total across cells with known cost: ${sum(known):,.2f} ({len(known)}/{len(USAGE)} cells)")
    STATUS["cost_usd_known"] = round(sum(known), 2)
else:
    print("[COST] nothing scored yet")

## 17. Report File

Writes `report.md` / `report.json` next to the results (and into `/kaggle/working` on Kaggle) from the state of this run, whatever happened before: a Run All never leaves a blank result. Every earlier step that failed or was skipped is listed with its reason.

In [ ]:
def build_report():
    g = globals()
    lines = ["# Frontier VLMs on Kaggle Benchmarks - run report", "", f"generated {time.strftime('%Y-%m-%d %H:%M:%S')} on {STATUS.get('platform')}", ""]
    checks = [("repository", "repo_sha", lambda v: bool(v)), ("dependencies", "dependencies_ok", bool), ("Hugging Face auth", "hf_ok", bool),
              ("Kaggle CLI", "kaggle_cli_ok", bool), ("preflight", "preflight_ok", bool), ("images on Kaggle", "images_on_kaggle", bool),
              ("smoke test", "smoke_ok", lambda v: v is True)]
    lines += ["## Checks", ""] + [f"- {name}: {'ok' if fn(STATUS.get(key)) else 'NOT OK / not run'}" for name, key, fn in checks]
    if STATUS.get("remote_sha") and STATUS.get("repo_sha") != STATUS.get("remote_sha"):
        lines.append(f"- WARNING: this checkout ({STATUS['repo_sha'][:10]}) differs from GitHub's default branch ({STATUS['remote_sha'][:10]}); hosted workers run the latter")
    lines += ["", "## Campaign", "", f"- state: {STATUS.get('campaign', 'not run')}",
              f"- images: {STATUS.get('n_images')} ({STATUS.get('split')}) | conditions: {STATUS.get('conditions')} | models: {STATUS.get('n_models')}"]
    rep = g.get("REPORT")
    if rep:
        lines += [f"- runs: {rep['runs']}", f"- paused: {rep['paused'][:5] or 'none'}"]
    lines += [f"- scored cells: {STATUS.get('scored_cells', 0)} | known cost: ${STATUS.get('cost_usd_known', 0)}", "", "## Comparison", "",
              g.get("COMPARISON_MD", "(not computed)")]
    return "\n".join(lines)


REPORT_TEXT = build_report()
REPORT_PATHS = [ROOT / "report.md"] + ([Path("/kaggle/working/report.md")] if IS_KAGGLE else [])
for path in REPORT_PATHS:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(REPORT_TEXT, encoding="utf-8")
(ROOT / "report.json").write_text(json.dumps({k: v for k, v in STATUS.items()}, indent=2, default=str), encoding="utf-8")
print(REPORT_TEXT)
print(f"\n[REPORT] written to {[str(p) for p in REPORT_PATHS]}")

## 18. How to Read and Report the Result

**What the table can support**
- *Tools vs zero-shot* and *rules+skills vs tools*, per model, are the harness-effectiveness numbers: paired on the same images, with a bootstrap interval. A difference whose interval spans zero is not a finding.
- Frontier models vs the trained checkpoint is a comparison of **systems** (model + harness + training), reported next to the checkpoint's own records (`EXTRA_EVALUATED`). The checkpoint's training labels differ from the 250-image test labels, which is exactly what the rules and skills are there to bridge; say so.
- Decoding: temperature 0 is requested and dropped for models that reject it (recorded per model in the run identity, `decoding`), so "greedy" is not guaranteed for every model; some thinking models also vary between runs.

**Things to state in the paper**
- Hosted runs use the CPU environment's tools: the YOLO grounder runs on CPU with the same weights (hash recorded per run); the model sees the same canonical views.
- Model prices here are indicative; the actual spend is the per-request usage in section 16.
- Failed or refused episodes count as failures (never dropped); `format_failure` and `error` outcomes are reported separately in the hindsight export.

**Resuming:** re-run sections 1-9 and 11-17 (the smoke test is skipped once a campaign exists). The campaign state and results come back from the Hub when `RESTORE_FROM_HUB` is on.